In [8]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain
from langchain_chroma import Chroma

import os

CHROMA_PATH = os.path.abspath(os.path.join("..", "db", "chroma_db"))

# Initialize the OpenAI embedding model
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=256 
)

# vector store
vector_store = Chroma(
    persist_directory=CHROMA_PATH,
    embedding_function=embeddings_model,
    collection_name='smartbnb_vector_store'
)

# create retriever
retriever = vector_store.as_retriever(
    search_kwargs={"k":5}
)

# fetch relevant documents
# retriever.invoke("Condesa")

# the building blocks
prompt = ChatPromptTemplate.from_template("""Answer the question based on the context below.
you must give to the user the listing in the documents context as recommendations. The user is looking 
for listings and make the reservation of the listing.

Context: {context}
                                            
Question: {question}

Answer:
""")

# Initialize the model
llm_model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# chain = prompt | llm_model

In [4]:
# 
@chain
def query_listings(input):
    # fetch relevant documents
    docs = retriever.get_relevant_documents("""codensa""")
    # format promtp
    formatted = prompt.invoke({"context":docs, "question":input})
    # generate answer
    return llm_model.invoke(formatted)

query_listings.invoke("Why i must choice this listings")

AIMessage(content='1. Amazing Pent House apartment in the heart of Condesa:\n- Price: 2839.0\n- Neighbourhood: Cuauhtémoc\n- Room type: Hotel room\n- Property type: Room in serviced apartment\n- Amenities: Free street parking, Free parking on premises, Hot water, TV with standard cable, Hangers, Exterior security cameras on property, Essentials, Dishwasher, Oven, Microwave, Coffee maker, Wifi, Washer, Dryer, Stove, Dishes and silverware, Hair dryer, Kitchen, Iron, Patio or balcony, Shampoo, Smoke alarm, Host greets you, Carbon monoxide alarm, Cooking basics, Backyard, Refrigerator\n- Minimum nights: 1\n- Maximum nights: 365\n- Bedrooms: 1\n- Bathrooms: 1 private bath\n- Review scores accuracy: 4.71\n\n2. Comfortably furnished, sunny, 2 bedroom apt., on the second floor, Colonia Condesa:\n- Price: 2182.0\n- Neighbourhood: Cuauhtémoc\n- Room type: Entire home/apt\n- Property type: Entire rental unit\n- Amenities: Sound system, Carbon monoxide alarm, Dedicated workspace, Record player, Ir

In [9]:
@chain
def query_listings(input):
    # fetch relevant documents
    docs = retriever.get_relevant_documents(input)
    # format promtp
    formatted = prompt.invoke({"context":docs, "question":input})
    # generate answer
    answer = llm_model.invoke(formatted)
    return {"answer":answer,"docs":docs}

query_listings.invoke("apartment near the university and public transport")

{'answer': AIMessage(content='Based on the context provided, here are some listings that match your criteria of being near the university and public transport:\n\n1. Document ID: 2373420\n   - Price: $571.0\n   - Room Type: Private room\n   - Neighbourhood: Coyoacán\n   - Amenities: Wifi, shared kitchen, living room, bathroom\n   - Close to major tourist spots, Ciudad Universitaria, Tlalpan, San Angelo\n   - Excellent public transport access\n\n2. Document ID: 6687737\n   - Price: $906.0\n   - Room Type: Private room\n   - Neighbourhood: Miguel Hidalgo\n   - Amenities: Wifi, shared kitchen, living room, 2 shared baths\n   - Centric area with subway stations nearby\n\nPlease review these listings and make a reservation based on your preferences.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 183, 'prompt_tokens': 2054, 'total_tokens': 2237, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens'

In [10]:
@chain
def query_listings(input):
    # fetch relevant documents
    docs = retriever.get_relevant_documents(input)
    # format promtp
    formatted = prompt.invoke({"context":docs, "question":input})
    # generate answer
    answer = llm_model.invoke(formatted)
    return {"answer":answer,"docs":docs}

query_listings.invoke("recommend me an apartments with roof")

{'answer': AIMessage(content='Based on the listings provided, I recommend the following apartment with a roof:\n\n1. Document ID: 395446\n   - Property Type: Entire condo\n   - Neighbourhood: Cuauhtémoc\n   - Price: 1693.0\n   - Amenities: Dining table, Free street parking, Hot water, Exterior security cameras, Hangers, Outdoor dining area, Essentials, Dishwasher, Oven, Microwave, Wifi, Shower gel, Toaster, Safe, Free dryer, Long term stays allowed, Freezer, Cleaning products, Dishes and silverware, Hair dryer, Fire extinguisher, Kitchen, Iron, Gas stove, Shampoo, and more.\n   - Description: This is a lovely one bedroom - one bathroom apartment with a super comfortable king size bed, a very fresh take on retro style, and an awesome rooftop. Enjoy spending time on the day-beds, and having fun with the Argentinian grill and wood-fired pizza oven.\n\nThis apartment offers a rooftop where you can relax and enjoy the views.', additional_kwargs={'refusal': None}, response_metadata={'token_u